In [2]:
import os,csv,re
import pandas as pd
import numpy as np
import scanpy as sc
import math
import SpaGCN as spg
from scipy.sparse import issparse
import random, torch
import warnings
warnings.filterwarnings("ignore")
import matplotlib.colors as clr
import matplotlib.pyplot as plt

folder = "nb"
result = pd.DataFrame(columns = ["ARI", "AMI", "NMI"])

for data_index in range(30):
    data = str(data_index + 1)
    file_name = folder + "/"+ "SpaGCN/data/" + data +".h5ad"
    adata = sc.read_h5ad(file_name)
    adata.X = adata.X.astype("float32")

    x_pixel=adata.obs["col"].tolist()
    y_pixel=adata.obs["row"].tolist()
    #Calculate adjacent matrix
    s=1
    b=49

    
    #If histlogy image is not available, SpaGCN can calculate the adjacent matrix using the function below
    adj=spg.calculate_adj_matrix(x=x_pixel,y=y_pixel, histology=False)

    #set hyper-parameters
    p=0.5 
    #Find the l value given p=
    l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)


    n_clusters = 4
    #print(n_clusters)
    #print(np.unique(obs_NA["Region"]))
    #Set seed
    r_seed=t_seed=n_seed=99
    #Seaech for suitable resolution
    res=spg.search_res(adata, adj, l, n_clusters, start=0.1, step=0.1, tol=5e-3, lr=0.05, max_epochs=20, r_seed=r_seed, t_seed=t_seed, n_seed=n_seed)

    #run SpaGCN
    clf=spg.SpaGCN()
    clf.set_l(l)
    #Set seed
    random.seed(r_seed)
    torch.manual_seed(t_seed)
    np.random.seed(n_seed)
    #Run
    clf.train(adata,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
    y_pred, prob=clf.predict()
           

    adata.obs["pred"]= y_pred
    adata.obs["pred"]=adata.obs["pred"].astype('category')

    from sklearn import metrics
    obs_df = adata.obs.dropna()
    ari = metrics.adjusted_rand_score(obs_df['pred'], obs_df['label'])
    nmi = metrics.normalized_mutual_info_score(obs_df['pred'], obs_df['label'])
    ami = metrics.adjusted_mutual_info_score(obs_df['pred'], obs_df['label'])

    values= [ari, ami, nmi]
    result.loc[data_index] = values

result_file = folder + "/summary/" + "SpaGCN.csv"
result.to_csv(result_file )
        

Calculateing adj matrix using xy only...
Run 1: l [0.01, 1000], p [0.0, 275.9861614091282]
Run 2: l [0.01, 500.005], p [0.0, 275.9446716308594]
Run 3: l [0.01, 250.0075], p [0.0, 275.77874755859375]
Run 4: l [0.01, 125.00874999999999], p [0.0, 275.116943359375]
Run 5: l [0.01, 62.509375], p [0.0, 272.4980773925781]
Run 6: l [0.01, 31.2596875], p [0.0, 262.4566345214844]
Run 7: l [0.01, 15.63484375], p [0.0, 228.24298095703125]
Run 8: l [0.01, 7.822421875], p [0.0, 148.19964599609375]
Run 9: l [0.01, 3.9162109375], p [0.0, 61.41034698486328]
Run 10: l [0.01, 1.9631054687499998], p [0.0, 18.57928466796875]
Run 11: l [0.01, 0.9865527343749999], p [0.0, 4.483887672424316]
Run 12: l [0.01, 0.49827636718749996], p [0.0, 0.5476319789886475]
Run 13: l [0.25413818359374996, 0.49827636718749996], p [0.0015752315521240234, 0.5476319789886475]
Run 14: l [0.37620727539062493, 0.49827636718749996], p [0.10892236232757568, 0.5476319789886475]
Run 15: l [0.4372418212890624, 0.49827636718749996], p [0.